# XGBoost: speed and missing values
Training time of classic gradient boosting against XGBoost (CPU and GPU), scikit-learn's histogram-based
gradient boosting and LightGBM; XGBoost on data with missing values; equal-width against quantile bins.

In [ ]:
import os

# use 4 processor cores in every library, so the comparison is fair
os.environ["OMP_NUM_THREADS"] = "4"

import time

import lightgbm as lgb
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.datasets import make_classification
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score, train_test_split
from xgboost import XGBClassifier

print("xgboost", xgb.__version__, "| lightgbm", lgb.__version__)

## 1. Training time on 10,000 rows and 200 columns

In [ ]:
# a synthetic dataset: 10,000 rows, 200 columns (20 of them informative), two classes
X, y = make_classification(n_samples=10_000, n_features=200, n_informative=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# every model: 100 trees, learning rate 0.1, trees of depth 3, so the comparison is about speed only
models = {
    # classic gradient boosting: every split value tried exactly, one core (it has no n_jobs)
    "sklearn GradientBoosting": GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42),
    # XGBoost on the CPU: histogram bins, all cores
    "XGBoost (CPU)": XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, tree_method="hist",
                                   device="cpu", n_jobs=4, random_state=42),
    # XGBoost on the graphics card
    "XGBoost (GPU)": XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, tree_method="hist",
                                   device="cuda", random_state=42),
    # scikit-learn's histogram-based gradient boosting (modelled on LightGBM)
    "sklearn HistGradientBoosting": HistGradientBoostingClassifier(max_iter=100, max_depth=3,
                                                                   early_stopping=False, random_state=42),
    # LightGBM
    "LightGBM": lgb.LGBMClassifier(n_estimators=100, max_depth=3, num_leaves=8, learning_rate=0.1,
                                   n_jobs=4, random_state=42, verbose=-1),
}
rows = []
for name, model in models.items():
    model.fit(X_train[:500], y_train[:500])          # warm-up (the first GPU call loads its libraries)
    start = time.perf_counter()
    model.fit(X_train, y_train)                      # time only the real training
    seconds = time.perf_counter() - start
    rows.append((name, round(seconds, 2), round(model.score(X_test, y_test), 3)))
timings = pd.DataFrame(rows, columns=["model", "seconds", "test accuracy"])
timings.to_csv("data/timings.csv", index=False)       # the Note's bar chart reads this file
timings

## 2. Missing values: the Titanic ages

In [ ]:
# 891 passengers; Age is missing for 177 of them
df = pd.read_csv("data/titanic_train.csv")
df["Sex"] = (df["Sex"] == "female").astype(int)      # 1 = female, 0 = male
X = df[["Pclass", "Sex", "Age", "Fare"]]
y = df["Survived"]
print(X.isna().sum())

In [ ]:
# classic gradient boosting refuses missing values
try:
    GradientBoostingClassifier().fit(X, y)
except ValueError as e:
    print("ValueError:", str(e).splitlines()[0])

In [ ]:
# XGBoost trains on them as they are (np.nan is its default marker for 'missing')
xgb_model = XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, missing=np.nan, random_state=42)
print("5-fold CV accuracy, NaN kept:     ", cross_val_score(xgb_model, X, y, cv=5).mean().round(3))

# for comparison: fill the missing ages with the median first
X_filled = X.fillna({"Age": X["Age"].median()})
print("5-fold CV accuracy, median-filled:", cross_val_score(xgb_model, X_filled, y, cv=5).mean().round(3))

In [ ]:
# the learned default direction: in the first tree, where do passengers with no Age go?
xgb_model.fit(X, y)
tree0 = xgb_model.get_booster().trees_to_dataframe().query("Tree == 0")
age_splits = tree0[tree0["Feature"] == "Age"][["ID", "Split", "Yes", "No", "Missing"]]
age_splits["missing goes"] = np.where(age_splits["Missing"] == age_splits["Yes"], "left (Age < split)",
                                      "right (Age >= split)")
age_splits

## 3. Equal-width bins and quantile bins on a skewed column

In [ ]:
fare = df["Fare"]
print(fare.describe().round(1))

# 8 bins of equal width
uniform_edges = np.linspace(fare.min(), fare.max(), 9)
# 8 bins with about the same number of passengers each (quantiles 0, 1/8, ..., 1)
quantile_edges = np.quantile(fare, np.linspace(0, 1, 9))

print("uniform edges: ", uniform_edges.round(1))
print("rows per bin:  ", np.histogram(fare, uniform_edges)[0])
print("quantile edges:", quantile_edges.round(1))
print("rows per bin:  ", np.histogram(fare, quantile_edges)[0])